# 01 — EDA Raw Data (Updated 2026-07-26)

Phân tích cấu trúc 2 dataset sau khi download:
- `glaiveai/glaive-function-calling-v2` — multi-turn chat (112,960 samples)
- `Salesforce/xlam-function-calling-60k` — flat function_call (60,000 samples)

**Findings chính** (xem chi tiết `data/raw/EDA_SUMMARY.md`):
- Glaive: 40% single-turn có function call (45,593 samples usable)
- xLAM: 47% single-call (28,461), 42% có 2 calls, 11% có 3+ calls
- Tool diversity: Glaive 1,040 tools (system field), xLAM 3,605 tools
- Glaive: FC format = `<functioncall> {"name": "...", "arguments": '{...}'} <|endoftext|>`
- xLAM: fields = `id, query, answers (JSON str list), tools (JSON str list)`

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

RAW_DIR = Path('data/raw')

## 1. Glaive — basic

In [ ]:
glaive = [json.loads(line) for line in (RAW_DIR / 'glaive_raw.jsonl').open(encoding='utf-8')]
print(f'Total: {len(glaive):,}')
print(f'Fields: {list(glaive[0].keys())}')

In [ ]:
print('=== Sample 0 — full ===')
print('SYSTEM:', glaive[0]['system'][:500])
print()
print('CHAT:', glaive[0]['chat'][:1000])

In [ ]:
for i in [0, 1, 2]:
    if '<functioncall>' in glaive[i].get('chat', ''):
        print(f'=== Sample {i} with function call ===')
        print('CHAT (first 2000):', glaive[i]['chat'][:2000])
        break

## 2. Glaive — statistics

In [ ]:
fc_pat = re.compile(r'<functioncall>\s*(\{.*?\})\s*<\|endoftext\|>', re.DOTALL)
single_turn_pat = re.compile(r'USER:\s*(.*?)(?=\n\s*(?:A:|ASSISTANT:))', re.DOTALL)
asst_pat = re.compile(r'(?:A:|ASSISTANT:)\s*(.*?)(?=\n\s*(?:USER:|FUNCTION RESPONSE:|$))', re.DOTALL)

stats = Counter()
for s in glaive:
    chat = s.get('chat', '')
    if '<functioncall>' in chat:
        stats['any_fc'] += 1
    if chat.count('USER:') > 1:
        stats['multi_turn'] += 1
    m_user = single_turn_pat.search(chat)
    m_a = asst_pat.search(chat)
    if m_user and m_a and '<functioncall>' in m_a.group(1):
        stats['first_turn_fc'] += 1

print(f"Total: {stats.get('any_fc', 0) + (len(glaive) - stats.get('any_fc', 0)):,}")
print(f"Any FC: {stats['any_fc']:,} ({100*stats['any_fc']/len(glaive):.1f}%)")
print(f"Multi-turn: {stats['multi_turn']:,} ({100*stats['multi_turn']/len(glaive):.1f}%)")
print(f"First-turn FC (USABLE): {stats['first_turn_fc']:,} ({100*stats['first_turn_fc']/len(glaive):.1f}%)")

In [ ]:
fc_names = Counter()
for s in glaive:
    for m in fc_pat.finditer(s.get('chat', '')):
        try:
            fc = json.loads(m.group(1))
            fc_names[fc.get('name', '?')] += 1
        except json.JSONDecodeError:
            pass
print(f'Unique function names: {len(fc_names):,}')
print('Top 20:')
for n, c in fc_names.most_common(20):
    print(f'  {n}: {c}')

## 3. xLAM — basic

In [ ]:
xlam = [json.loads(line) for line in (RAW_DIR / 'xlam_raw.jsonl').open(encoding='utf-8')]
print(f'Total: {len(xlam):,}')
print(f'Fields: {list(xlam[0].keys())}')

In [ ]:
print('=== Sample 0 ===')
print(json.dumps(xlam[0], ensure_ascii=False, indent=2))

In [ ]:
print('=== Sample 1 ===')
print(json.dumps(xlam[1], ensure_ascii=False, indent=2))

## 4. xLAM — statistics

In [ ]:
calls_per_query = Counter()
for s in xlam:
    answers = json.loads(s.get('answers', '[]'))
    calls_per_query[len(answers)] += 1

print('xLAM: tool calls per query')
for n in sorted(calls_per_query.keys()):
    if calls_per_query[n] > 100:
        print(f'  {n} call(s): {calls_per_query[n]:,} ({100*calls_per_query[n]/len(xlam):.1f}%)')
    elif n <= 5 or calls_per_query[n] > 0:
        print(f'  {n} call(s): {calls_per_query[n]:,}')
print(f'\nTotal unique tools: {len({t["name"] for s in xlam for t in json.loads(s.get("tools", "[]"))}):,}')

In [ ]:
import statistics
ql = [len(s.get('query', '')) for s in xlam]
print(f'xLAM query length: min={min(ql)}, max={max(ql)}, mean={statistics.mean(ql):.1f}, median={statistics.median(ql):.0f}')

## 5. Tổng kết & open questions

**Usable samples sau filter**:
- Glaive single-turn + FC: **45,593** (40% of raw)
- xLAM single-call: **28,461** (47% of raw) — *nếu chỉ lấy first call*
- **MVP total: ~74k samples** (sau khi normalize)

**Open questions** (xem `data/raw/EDA_SUMMARY.md` §6):

1. Sample size cho pilot translate: 2k? Full 74k?
2. `feature_group` cho dataset không có sẵn: dùng default?
3. Multi-call trong xLAM: lấy first only hay expand thành N samples?
4. Quality filter cho Glaive: 40% còn lại OK hay thêm filter?

Sẽ hỏi user trước khi viết `normalize_schema.py`.